In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q timm

import os, json, random, copy, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
import timm
import shutil

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

TEA_LEAF_ROOT = "/content/drive/MyDrive/Tea Leaf Dataset"
OUT = "/content/drive/MyDrive/TeaLeaf_DINOv2_Study"
RESULTS_DIR = f"{OUT}/results"
FIGURES_DIR = f"{OUT}/figures"

# Copy to local disk for speed
LOCAL_DATA = "/tmp/tea_leaf_local"
if not os.path.isdir(LOCAL_DATA):
    print("Copying dataset to local disk (~2 min)...")
    shutil.copytree(TEA_LEAF_ROOT, LOCAL_DATA)
    print("✅ Copied")
else:
    print("✅ Local cache exists")

classes = sorted([d for d in os.listdir(LOCAL_DATA)
                  if os.path.isdir(os.path.join(LOCAL_DATA, d))])
NUM_CLASSES = len(classes)
CLASS_TO_IDX = {c: i for i, c in enumerate(classes)}
healthy_classes = [c for c in classes if 'healthy' in c.lower()]

# Load DINOv1
dv1_path = os.path.join(OUT, 'checkpoints', 'dino_vits16.pth')
if not os.path.exists(dv1_path):
    os.makedirs(os.path.dirname(dv1_path), exist_ok=True)
    !wget -q -O {dv1_path} https://dl.fbaipublicfiles.com/dino/dino_deitsmall16_pretrain/dino_deitsmall16_pretrain.pth

dino_v1 = timm.create_model('vit_small_patch16_224', pretrained=False, num_classes=0)
ckpt = torch.load(dv1_path, map_location='cpu')
sd = {k.replace('module.','').replace('backbone.','').replace('encoder.',''): v
      for k, v in ckpt.items() if 'head' not in k}
dino_v1.load_state_dict(sd, strict=False)
dino_v1.eval().to(device)
for p in dino_v1.parameters():
    p.requires_grad = False

_dv1_store = {}
dv1_last_attn = dino_v1.blocks[-1].attn
def _dv1_patched_forward(x, **kwargs):
    B, N, C = x.shape
    qkv = dv1_last_attn.qkv(x).reshape(B, N, 3, dv1_last_attn.num_heads, C // dv1_last_attn.num_heads)
    qkv = qkv.permute(2, 0, 3, 1, 4)
    q, k, v = qkv.unbind(0)
    scale = (C // dv1_last_attn.num_heads) ** -0.5
    cls_q = q[:, :, 0:1, :]
    cls_attn = (cls_q @ k.transpose(-2, -1)) * scale
    cls_attn = cls_attn.softmax(dim=-1)
    _dv1_store['cls_attn'] = cls_attn[:, :, 0, 1:].detach().cpu()
    attn_full = (q @ k.transpose(-2, -1)) * scale
    attn_full = attn_full.softmax(dim=-1)
    x = (attn_full @ v).transpose(1, 2).reshape(B, N, C)
    x = dv1_last_attn.proj(x)
    x = dv1_last_attn.proj_drop(x)
    return x
dv1_last_attn.forward = _dv1_patched_forward

tf_dv1 = T.Compose([
    T.Resize((224, 224)), T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
tf_train = T.Compose([
    T.Resize((224, 224)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.RandomRotation(20), T.ColorJitter(0.2, 0.2, 0.2),
    T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
tf_test = T.Compose([
    T.Resize((224, 224)), T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

DV1_GRID = 14
N_HEADS = 6
topk = max(1, int(0.1 * DV1_GRID * DV1_GRID))

# Collect all images (LOCAL paths)
all_images = []
for cls_name in classes:
    cls_dir = os.path.join(LOCAL_DATA, cls_name)
    for fname in sorted(os.listdir(cls_dir)):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            all_images.append({
                'path': os.path.join(cls_dir, fname),
                'class': cls_name,
                'class_idx': CLASS_TO_IDX[cls_name],
                'is_diseased': 0 if cls_name in healthy_classes else 1,
            })

# Fixed split
random.seed(42)
shuffled = all_images.copy()
random.shuffle(shuffled)
n_test = int(0.2 * len(shuffled))
test_images = shuffled[:n_test]
train_images = shuffled[n_test:]

# Helper datasets
class SimpleDS(Dataset):
    def __init__(self, imgs, labels, transform):
        self.imgs = imgs; self.labels = labels; self.transform = transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, idx):
        img = Image.open(self.imgs[idx]['path']).convert('RGB')
        return self.transform(img), self.labels[idx]

class TestDS(Dataset):
    def __init__(self, imgs, transform):
        self.imgs = imgs; self.transform = transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, idx):
        img = Image.open(self.imgs[idx]['path']).convert('RGB')
        return self.transform(img), self.imgs[idx]['is_diseased']

# Check what's already done
print(f"\n✅ Ready: {len(all_images)} images, {NUM_CLASSES} classes")
print(f"   Train: {len(train_images)}, Test: {len(test_images)}")
print(f"\nExisting results on Drive:")
for f in sorted(os.listdir(RESULTS_DIR)):
    if f.endswith('.json'):
        print(f"   ✅ {f}")

In [ ]:
# Multi-run pipeline: 3 seeds, report mean±std
# Saves after completion — ~45 min total

print("="*60)
print("EXPERIMENT: Multi-Run Pipeline (3 seeds)")
print("="*60)

# Check if already done
multi_path = os.path.join(RESULTS_DIR, 'multi_run_pipeline.json')
if os.path.exists(multi_path):
    with open(multi_path) as f:
        saved = json.load(f)
    print(f"Already done! Summary:")
    s = saved['summary']
    print(f"  AUC: {s['auc_mean']:.4f} ± {s['auc_std']:.4f}")
    print(f"  Acc: {s['acc_mean']:.4f} ± {s['acc_std']:.4f}")
    print(f"  F1:  {s['f1_mean']:.4f} ± {s['f1_std']:.4f}")
else:
    N_SEEDS = 3
    multi_results = []

    for seed in range(N_SEEDS):
        print(f"\n--- Seed {seed+1}/{N_SEEDS} ---")
        random.seed(seed * 1000 + 42)

        # Different split each seed
        shuf = all_images.copy()
        random.shuffle(shuf)
        n_t = int(0.2 * len(shuf))
        test_s = shuf[:n_t]
        train_s = shuf[n_t:]

        # UAHS head discovery
        pool = random.sample(train_s, min(200, len(train_s)))
        head_ent = {h: [] for h in range(N_HEADS)}
        with torch.no_grad():
            for sample in pool:
                try:
                    img = Image.open(sample['path']).convert('RGB')
                    _ = dino_v1(tf_dv1(img).unsqueeze(0).to(device))
                    attn = _dv1_store['cls_attn'][0].numpy()
                    for h in range(N_HEADS):
                        a = attn[h]; a = a/(a.sum()+1e-10)
                        head_ent[h].append(-(a*np.log(a+1e-10)).sum())
                except: continue
        sorted_heads = sorted(range(N_HEADS), key=lambda h: -np.std(head_ent[h]))
        sel_heads = sorted_heads[:2]
        print(f"  UAHS heads: {sel_heads}")

        # Score all training images
        scores = []
        with torch.no_grad():
            for s in tqdm(train_s, desc=f"  Scoring seed {seed+1}"):
                try:
                    img = Image.open(s['path']).convert('RGB')
                    _ = dino_v1(tf_dv1(img).unsqueeze(0).to(device))
                    attn = _dv1_store['cls_attn'][0].numpy()
                    a = attn[sel_heads].mean(axis=0)
                    a_n = a / (a.sum() + 1e-10)
                    scores.append(np.sort(a_n)[::-1][:topk].sum())
                except: scores.append(0.0)

        scores = np.array(scores)
        true_bin = np.array([s['is_diseased'] for s in train_s])

        # Pseudo-labels
        h_frac = (true_bin == 0).mean()
        thresh = np.percentile(scores, h_frac * 100)
        pseudo = (scores > thresh).astype(int)
        p_acc = accuracy_score(true_bin, pseudo)
        p_f1 = f1_score(true_bin, pseudo)
        print(f"  Pseudo acc: {p_acc:.4f}, F1: {p_f1:.4f}")

        # Train MobileNetV3
        ds_t = SimpleDS(train_s, pseudo.tolist(), tf_train)
        ds_e = TestDS(test_s, tf_test)
        loader_t = DataLoader(ds_t, batch_size=64, shuffle=True, num_workers=2)
        loader_e = DataLoader(ds_e, batch_size=64, shuffle=False, num_workers=2)
        test_true_s = np.array([s['is_diseased'] for s in test_s])

        model = models.mobilenet_v3_small(weights='IMAGENET1K_V1')
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 2)
        model = model.to(device)
        opt = optim.Adam(model.parameters(), lr=1e-3)
        sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=30)
        crit = nn.CrossEntropyLoss()

        model.train()
        for ep in range(30):
            for imgs, labels in loader_t:
                imgs, labels = imgs.to(device), labels.to(device).long()
                loss = crit(model(imgs), labels)
                opt.zero_grad(); loss.backward(); opt.step()
            sched.step()
            if (ep+1) % 10 == 0:
                print(f"    Epoch {ep+1}/30")

        model.eval()
        probs_all, preds_all = [], []
        with torch.no_grad():
            for imgs, tb in loader_e:
                out = model(imgs.to(device))
                probs_all.append(torch.softmax(out, 1)[:, 1].cpu().numpy())
                preds_all.append(out.argmax(1).cpu().numpy())
        probs_all = np.concatenate(probs_all)
        preds_all = np.concatenate(preds_all)

        auc = roc_auc_score(test_true_s, probs_all)
        acc = accuracy_score(test_true_s, preds_all)
        f1 = f1_score(test_true_s, preds_all)

        print(f"  ★ AUC={auc:.4f}, Acc={acc:.4f}, F1={f1:.4f}")

        multi_results.append({
            'seed': seed, 'uahs_heads': sel_heads,
            'pseudo_acc': float(p_acc), 'pseudo_f1': float(p_f1),
            'pipeline_auc': float(auc), 'pipeline_acc': float(acc),
            'pipeline_f1': float(f1),
        })
        del model; torch.cuda.empty_cache()

    # Summary
    aucs = [r['pipeline_auc'] for r in multi_results]
    accs = [r['pipeline_acc'] for r in multi_results]
    f1s = [r['pipeline_f1'] for r in multi_results]
    paccs = [r['pseudo_acc'] for r in multi_results]

    print(f"\n{'='*60}")
    print("MULTI-RUN SUMMARY (mean ± std)")
    print(f"{'='*60}")
    print(f"  Pipeline AUC:     {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
    print(f"  Pipeline Acc:     {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"  Pipeline F1:      {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    print(f"  Pseudo-label Acc: {np.mean(paccs):.4f} ± {np.std(paccs):.4f}")

    with open(multi_path, 'w') as f:
        json.dump({
            'runs': multi_results,
            'summary': {
                'auc_mean': float(np.mean(aucs)), 'auc_std': float(np.std(aucs)),
                'acc_mean': float(np.mean(accs)), 'acc_std': float(np.std(accs)),
                'f1_mean': float(np.mean(f1s)), 'f1_std': float(np.std(f1s)),
                'pseudo_acc_mean': float(np.mean(paccs)), 'pseudo_acc_std': float(np.std(paccs)),
            }
        }, f, indent=2)
    print("✅ Saved")

In [ ]:
# Formal inference cost table  (fixed: no hook/deepcopy device bug)
import os, json, time
import torch
import torchvision.models as models
import timm

print("="*60)
print("EXPERIMENT: Inference Cost")
print("="*60)

infer_path = os.path.join(RESULTS_DIR, 'inference_cost_formal.json')
if os.path.exists(infer_path):
    with open(infer_path) as f:
        saved = json.load(f)
    print("Already done!")
    for name, d in saved.items():
        print(f"  {name}: {d['params_M']}M params, GPU={d['gpu_ms']}ms, CPU={d['cpu_ms']}ms")
else:
    HAS_CUDA = torch.cuda.is_available()

    def make_mobilenet():
        m = models.mobilenet_v3_small(weights='IMAGENET1K_V1')
        m.classifier[-1] = torch.nn.Linear(m.classifier[-1].in_features, 2)  # deployed 2-class head
        return m.eval()

    # fresh, un-hooked instances — timing/params depend only on architecture
    factories = {
        'DINOv1 ViT-S/16'  : lambda: timm.create_model('vit_small_patch16_224',
                                                        pretrained=False, num_classes=0).eval(),
        'MobileNetV3-Small': make_mobilenet,
        'EfficientNet-B0'  : lambda: models.efficientnet_b0(weights='IMAGENET1K_V1').eval(),
        'ResNet-18'        : lambda: models.resnet18(weights='IMAGENET1K_V1').eval(),
    }

    # optional FLOPs
    try:
        from fvcore.nn import FlopCountAnalysis
        import logging; logging.getLogger("fvcore").setLevel(logging.ERROR)
        HAS_FVCORE = True
    except Exception:
        try:
            import subprocess, sys
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fvcore"], check=True)
            from fvcore.nn import FlopCountAnalysis
            import logging; logging.getLogger("fvcore").setLevel(logging.ERROR)
            HAS_FVCORE = True
        except Exception:
            HAS_FVCORE = False
            print("(fvcore unavailable — FLOPs will show n/a)")

    def bench_one(factory):
        d_cpu = torch.randn(1, 3, 224, 224)
        # params / size on a CPU instance
        m_cpu = factory().cpu()
        n_p = sum(p.numel() for p in m_cpu.parameters())
        sz  = sum(p.nelement() * p.element_size() for p in m_cpu.parameters()) / 1e6

        # FLOPs (MACs) on CPU instance
        if HAS_FVCORE:
            try:
                gflops = round(FlopCountAnalysis(m_cpu, d_cpu).total() / 1e9, 2)
            except Exception:
                gflops = None
        else:
            gflops = None

        # CPU latency
        with torch.no_grad():
            for _ in range(3): m_cpu(d_cpu)
            t0 = time.time()
            for _ in range(20): m_cpu(d_cpu)
            cpu_ms = (time.time() - t0) / 20 * 1000

        # GPU latency (separate GPU instance)
        if HAS_CUDA:
            m_gpu = factory().cuda()
            d_gpu = d_cpu.cuda()
            with torch.no_grad():
                for _ in range(10): m_gpu(d_gpu)
                torch.cuda.synchronize()
                t0 = time.time()
                for _ in range(50): m_gpu(d_gpu)
                torch.cuda.synchronize()
                gpu_ms = (time.time() - t0) / 50 * 1000
            del m_gpu, d_gpu; torch.cuda.empty_cache()
        else:
            gpu_ms = float('nan')

        del m_cpu, d_cpu
        return n_p, sz, gpu_ms, cpu_ms, gflops

    print(f"\n{'Model':<22s} {'Params':>8s} {'MB':>8s} {'GFLOPs':>8s} "
          f"{'GPU ms':>8s} {'CPU ms':>8s} {'FPS':>6s}")
    print("-" * 72)

    results = {}
    for name, factory in factories.items():
        n_p, sz, gpu_ms, cpu_ms, gflops = bench_one(factory)
        fps = 1000 / cpu_ms
        results[name] = {
            'params_M': round(n_p/1e6, 2), 'size_MB': round(sz, 1),
            'gflops': gflops, 'gpu_ms': round(gpu_ms, 1),
            'cpu_ms': round(cpu_ms, 1), 'fps_cpu': round(fps, 1),
        }
        gf = f"{gflops:>8.2f}" if gflops is not None else f"{'n/a':>8s}"
        print(f"{name:<22s} {n_p/1e6:>7.2f}M {sz:>7.1f} {gf} "
              f"{gpu_ms:>7.1f} {cpu_ms:>7.1f} {fps:>5.1f}")

    comp  = results['DINOv1 ViT-S/16']['params_M'] / results['MobileNetV3-Small']['params_M']
    speed = results['DINOv1 ViT-S/16']['cpu_ms']   / results['MobileNetV3-Small']['cpu_ms']
    print(f"\n  DINOv1→MobileNetV3: {comp:.1f}× compression, {speed:.1f}× CPU speedup")

    with open(infer_path, 'w') as f:
        json.dump(results, f, indent=2)
    if HAS_CUDA: torch.cuda.empty_cache()
    print("Saved")

In [ ]:
# Formal hyperparameter table
print("="*60)
print("Hyperparameter Configuration")
print("="*60)

hyperparams = {
    'DINOv1_Feature_Extraction': {
        'Model': 'DINOv1 ViT-S/16',
        'Input_Resolution': '224x224',
        'Patch_Size': 16,
        'Grid_Size': '14x14 (196 patches)',
        'Feature_Dim': 384,
        'Attention_Heads': 6,
        'Pretrained': 'DINO self-distillation (ImageNet-1K)',
        'Frozen': True,
    },
    'UAHS_Head_Selection': {
        'Criterion': 'Entropy variance (std of per-image entropy)',
        'Pool_Size': 200,
        'Auto_K': 'Marginal AUC gain > 0.002',
        'Selected_K': 2,
    },
    'Pseudo_Label_Generation': {
        'Scoring': 'Top-10% attention patch mass (concentration)',
        'Thresholding': 'Percentile (healthy class fraction)',
        'Alternative': 'GMM 2-component',
    },
    'MobileNetV3_Training': {
        'Architecture': 'MobileNetV3-Small',
        'Pretrained': 'ImageNet-1K',
        'Head': 'Linear(576, 2)',
        'Optimizer': 'Adam (lr=1e-3)',
        'Scheduler': 'CosineAnnealing (T_max=30)',
        'Epochs': 30,
        'Batch_Size': 64,
        'Loss': 'CrossEntropyLoss',
        'Augmentation': 'HFlip, VFlip, Rotation(20), ColorJitter(0.2)',
    },
}

for section, params in hyperparams.items():
    print(f"\n  {section}:")
    for k, v in params.items():
        print(f"    {k}: {v}")

with open(os.path.join(RESULTS_DIR, 'hyperparameters.json'), 'w') as f:
    json.dump(hyperparams, f, indent=2, default=str)
print("\n✅ Saved")